# Notebook: `06. Silver`
---

## Descrição: 

Esse notebook realiza o tratamento dos dados na camada `bronze` utilizando as regras de qualidade definidas em `05. Data Quality`, criação de colunas, tipagem de dados e inserções mínimas de registros, como interpolação de valores ausentes nas tabelas de inflação.

## Origem

- **yellowTaxiTripData**: `mvp.bronze.yellowTaxiTripData`;
- **greenTaxiTripData**: `mvp.bronze.greenTaxiTripData`;
- **forHireVehicles**: `mvp.bronze.forHireVehicles`
- **forHireVehiclesHighVolume**: `mvp.bronze.forHireVehiclesHighVolume`
- **taxi_zone_lookup**: `mvp.bronze.taxi_zone_lookup`
- **fhv_base_lookup**: `mvp.bronze.fhv_base_lookup`
- **Feriados_US_NY**: `mvp.bronze.Feriados_US_NY`
- **Inflacao_CPI_U**: `mvp.bronze.Inflacao_CPI_U`
- **Inflacao_CPI_T**: `mvp.bronze.Inflacao_CPI_T`

## Destino

- **yellowTaxiTripData**: `mvp.silver.yellowTaxiTripData`;
- **greenTaxiTripData**: `mvp.silver.greenTaxiTripData`;
- **forHireVehicles**: `mvp.silver.forHireVehicles`
- **forHireVehiclesHighVolume**: `mvp.silver.forHireVehiclesHighVolume`
- **taxi_zone_lookup**: `mvp.silver.taxi_zone_lookup`
- **fhv_base_lookup**: `mvp.silver.fhv_base_lookup`
- **Feriados_US_NY**: `mvp.silver.Feriados_US_NY`
- **Inflacao_CPI_U**: `mvp.silver.Inflacao_CPI_U`
- **Inflacao_CPI_T**: `mvp.silver.Inflacao_CPI_T`

Definição do catálogo e schema usado

In [0]:
%sql
USE CATALOG mvp;
USE SCHEMA silver;

Impoprtação das bibliotecas usadas.

In [0]:
from pyspark.sql.functions import col, to_date, hour, year, month, concat_ws, lpad, try_divide, trim, regexp_replace, max, lag, lead, when, lit
from pyspark.sql.window import Window

### silver.yellowtaxitripdata (Táxi Amarelo)

Leitura da tabela `mvp.bronze.yellowtaxitripdata`, aplicação dos filtros de qualidade de dados e das transformações de data e hora, e gravação do resultado na tabela Delta `mvp.silver.yellowtaxitripdata`. As justificativas de cada filtro estão detalhadas na seção de Qualidade de Dados.

**Seleção de colunas**: da tabela bronze são mantidas apenas as colunas necessárias às perguntas de negócio: `VendorID`, os timestamps de embarque e desembarque, `PULocationID`, `DOLocationID`, `RatecodeID`, `payment_type`, `fare_amount`, `tip_amount`, `trip_distance` (usada apenas nos filtros, não persistida na silver) e `_source_file_yearmonth` (usada apenas na validação de data, também não persistida). As demais colunas de tarifa e taxas são descartadas nesta etapa.

**Transformações aplicadas:**

1. **Separação de data e hora**: cada timestamp é decomposto em uma coluna de data (`tpep_pickup_date`, `tpep_dropoff_date`) e uma coluna com a hora inteira (`pickup_hour`, `dropoff_hour`), preparando os dados para as agregações por dia e hora da camada gold.

**Filtros de qualidade aplicados** (registros que não atendem a todos são descartados):

1. **Consistência temporal**: o horário de desembarque deve ser posterior ao de embarque.
2. **Consistência com o arquivo de origem**: o ano e mês da data de embarque devem coincidir com o ano e mês do arquivo mensal do qual o registro foi lido (`_source_file_yearmonth`), eliminando os registros com datas fora do período do arquivo (como os timestamps inválidos de anos como 2001 ou 2098 identificados na bronze).
3. **Valores monetários e distância**: `trip_distance` e `fare_amount` não podem ser negativos, e a razão entre tarifa e distância (`fare_amount / trip_distance`) não pode exceder 30,75 dólares por milha (regra IEQ para remoção de outliers, descrita na qualidade dos dados). Viagens com distância zero geram divisão inválida, e `try_divide` retorna `NULL` nesses casos, o que faz o filtro descartá-las.
4. **Gorjeta**: valores nulos são mantidos, mas gorjetas negativas ou superiores a 100 dólares são descartadas.
5. **Código de tarifa**: registros com `RatecodeID` nulo são descartados.
6. **Localização desconhecida**: registros com embarque ou desembarque no código de zona 264 ("Desconhecido") são descartados.

**Comando final de escrita:**

```python
df_final.write.mode("overwrite").saveAsTable("mvp.silver.yellowtaxitripdata")
```

In [0]:
df_yellow = spark.table("mvp.bronze.yellowtaxitripdata")

df_yellow = df_yellow.select("VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime", 
                             "PULocationID", "DOLocationID", "RatecodeID", "payment_type", "fare_amount", "tip_amount", "trip_distance", "_source_file_yearmonth")

df_enriquecido = (
    df_yellow
    .withColumn("tpep_pickup_date", to_date(col("tpep_pickup_datetime")))
    .withColumn("tpep_dropoff_date", to_date(col("tpep_dropoff_datetime")))
    .withColumn("pickup_hour", hour(col("tpep_pickup_datetime")))
    .withColumn("dropoff_hour", hour(col("tpep_dropoff_datetime")))
)

df_filtrado = (
    df_enriquecido
    .filter(col("tpep_dropoff_datetime") > col("tpep_pickup_datetime"))
    .filter(
        concat_ws("-",
            year(col("tpep_pickup_datetime")).cast("string"),
            lpad(month(col("tpep_pickup_datetime")).cast("string"), 2, "0")
        ) == col("_source_file_yearmonth")
    )
    .filter( (col("trip_distance") >= 0 ) & ( col("fare_amount") >= 0 ) & ( try_divide( col("fare_amount"), col("trip_distance") ) <= 30.75))
    .filter(
        col("tip_amount").isNull() | 
        ( (col("tip_amount") >= 0)  & (col("tip_amount") <= 100) )
    )
    .filter(col("RatecodeID").isNotNull())
    .filter( (col("PUlocationID") != 264) & (col("DOlocationID") != 264))
)

df_final = (
    df_filtrado.select("VendorID", "tpep_pickup_date", "tpep_dropoff_date", "pickup_hour", "dropoff_hour",
                       "PULocationID", "DOLocationID", "RatecodeID", "payment_type", "fare_amount",
                       "tip_amount")
)

df_final.write.mode("overwrite").saveAsTable("mvp.silver.yellowtaxitripdata")


### silver.greentaxitripdata (Táxi Verde)

Leitura da tabela `mvp.bronze.greentaxitripdata`, aplicação dos filtros de qualidade de dados e das transformações de data e hora, e gravação do resultado na tabela Delta `mvp.silver.greentaxitripdata`. A estrutura é análoga à da silver do táxi amarelo, com as diferenças descritas abaixo. As justificativas de cada filtro estão detalhadas na seção de Qualidade de Dados.

**Seleção de colunas**: da tabela bronze são mantidas apenas as colunas necessárias às perguntas de negócio: `VendorID`, os timestamps de embarque e desembarque (`lpep_pickup_datetime`, `lpep_dropoff_datetime`), `PULocationID`, `DOLocationID`, `RatecodeID`, `payment_type`, `fare_amount`, `tip_amount`, `trip_distance` (usada apenas nos filtros, não persistida na silver) e `_source_file_yearmonth` (usada apenas na validação de data, também não persistida). As demais colunas de tarifa e taxas são descartadas nesta etapa.

**Transformações aplicadas:**

1. **Separação de data e hora**: cada timestamp é decomposto em uma coluna de data (`lpep_pickup_date`, `lpep_dropoff_date`) e uma coluna com a hora inteira (`pickup_hour`, `dropoff_hour`), preparando os dados para as agregações por dia e hora da camada gold.

**Filtros de qualidade aplicados** (registros que não atendem a todos são descartados):

1. **Consistência temporal**: o horário de desembarque deve ser posterior ao de embarque.
2. **Consistência com o arquivo de origem**: o ano e mês da data de embarque devem coincidir com o ano e mês do arquivo mensal do qual o registro foi lido (`_source_file_yearmonth`), eliminando os registros com datas fora do período do arquivo.
3. **Gorjeta**: valores nulos são mantidos, mas gorjetas negativas ou superiores a 100 dólares são descartadas.
4. **Valores monetários e distância**: `trip_distance` e `fare_amount` não podem ser negativos, e a razão entre tarifa e distância (`fare_amount / trip_distance`) não pode exceder 31 dólares por milha (regra IEQ para remoção de outliers). Viagens com distância zero geram divisão inválida, e `try_divide` retorna `NULL` nesses casos, o que faz o filtro descartá-las.
5. **Localização desconhecida**: registros com embarque e desembarque no código de zona 264 ("Desconhecido") são descartados.
6. **Código de tarifa e método de pagamento**: registros com `RatecodeID` ou `payment_type` nulos são descartados.

**Comando final de escrita:**

```python
df_final.write.mode("overwrite").saveAsTable("mvp.silver.greentaxitripdata")
```

In [0]:
df_green = spark.table("mvp.bronze.greentaxitripdata")

df_green = df_green.select("VendorID", "lpep_pickup_datetime", "lpep_dropoff_datetime", 
                             "PULocationID", "DOLocationID", "RatecodeID", "payment_type", "fare_amount", "tip_amount", "trip_distance", "_source_file_yearmonth")

df_enriquecido = (
    df_green
    .withColumn("lpep_pickup_date", to_date(col("lpep_pickup_datetime")))
    .withColumn("lpep_dropoff_date", to_date(col("lpep_dropoff_datetime")))
    .withColumn("pickup_hour", hour(col("lpep_pickup_datetime")))
    .withColumn("dropoff_hour", hour(col("lpep_dropoff_datetime")))
)

df_filtrado = (
    df_enriquecido
    .filter(col("lpep_dropoff_datetime") > col("lpep_pickup_datetime"))
    .filter(
        concat_ws("-",
            year(col("lpep_pickup_datetime")).cast("string"),
            lpad(month(col("lpep_pickup_datetime")).cast("string"), 2, "0")
        ) == col("_source_file_yearmonth")
    )
    .filter(
        col("tip_amount").isNull() | 
        ( (col("tip_amount") >= 0)  & (col("tip_amount") <= 100) )
    )
    .filter( (col("trip_distance") >= 0 ) & (col("fare_amount") >= 0 ) & ( try_divide(col("fare_amount"), col("trip_distance") ) <= 31 ))
    .filter( (col("PUlocationID") != 264) & ( col("DOlocationID") != 264))
    .filter(col("RatecodeID").isNotNull())
    .filter(col("payment_type").isNotNull())
)

df_final = (
    df_filtrado.select("VendorID", "lpep_pickup_date", "lpep_dropoff_date", "pickup_hour", "dropoff_hour",
                       "PULocationID", "DOLocationID", "RatecodeID", "payment_type", "fare_amount",
                       "tip_amount")
)

df_final.write.mode("overwrite").saveAsTable("mvp.silver.greentaxitripdata")

### silver.forhirevehicles (For-Hire Vehicles, FHV)

Leitura da tabela `mvp.bronze.forhirevehicles`, aplicação dos filtros de qualidade de dados e das transformações de data e hora, e gravação do resultado na tabela Delta `mvp.silver.forhirevehicles`. Esta tabela é a fonte do histórico de viagens por aplicativo anteriores a fev/2019 (período em que ainda não existia a base HVFHS), e por isso é limitada até jan/2019. Diferentemente das silvers de táxi, parte dos filtros é condicional a uma **data de corte (01/01/2018)**, pois, conforme descrito na seção de Modelagem, os campos de desembarque e localização só passam a ser registrados de forma consistente a partir de meados de 2017 a TLC não informa a data exata de início da exigência. As justificativas de cada filtro estão detalhadas na seção de Qualidade de Dados.

**Seleção de colunas**: da tabela bronze são mantidas apenas as colunas necessárias: `pickup_datetime`, `dropOff_datetime`, `PUlocationID`, `DOlocationID`, `dispatching_base_num` (usada posteriormente na gold para associar a operadora via `fhv_base_lookup`) e `_source_file_yearmonth` (usada apenas na validação de data, não persistida na silver). As colunas `Affiliated_base_number` e `SR_Flag` são descartadas.

**Transformações aplicadas:**

1. **Separação de data e hora**: cada timestamp é decomposto em uma coluna de data (`pickup_date`, `dropoff_date`) e uma coluna com a hora inteira (`pickup_hour`, `dropoff_hour`), preparando os dados para as agregações por dia e hora da camada gold.

**Filtros de qualidade aplicados** (registros que não atendem a todos são descartados). Os filtros marcados como *condicionais* só se aplicam a viagens com embarque em 01/01/2018 ou depois; viagens anteriores a essa data passam por eles sem restrição:

1. **Consistência temporal** *(condicional)*: o horário de desembarque deve ser posterior ao de embarque.
2. **Consistência com o arquivo de origem**: o ano e mês da data de embarque devem coincidir com o ano e mês do arquivo mensal do qual o registro foi lido (`_source_file_yearmonth`).
3. **Limite superior de data**: são mantidos apenas registros com embarque até 31/01/2019, já que, a partir de fev/2019, as viagens de aplicativos de alto volume passam a ser registradas na base HVFHS.
4. **Códigos de zona válidos** *(condicional)*: `PUlocationID` e `DOlocationID` devem estar entre 1 e 264, descartando códigos fora do intervalo válido da tabela `taxi_zone_lookup`.
5. **Localização desconhecida ou externa** *(condicional)*: exclusão de registros nos códigos 264 ("Desconhecido") e 265 ("Fora de Nova Iorque").
6. **Localização nula** *(condicional)*: `PUlocationID` e `DOlocationID` não podem ser nulos.

**Comando final de escrita:**

```python
df_final.write.mode("overwrite").saveAsTable("mvp.silver.forhirevehicles")
```

In [0]:
df_fhv = spark.table("mvp.bronze.forhirevehicles")

df_fhv = df_fhv.select("pickup_datetime", "dropOff_datetime", 
                             "DOlocationID", "PUlocationID", "dispatching_base_num", "_source_file_yearmonth")

df_enriquecido = (
    df_fhv
    .withColumn("pickup_date", to_date(col("pickup_datetime")))
    .withColumn("dropoff_date", to_date(col("dropOff_datetime")))
    .withColumn("pickup_hour", hour(col("pickup_datetime")))
    .withColumn("dropoff_hour", hour(col("dropOff_datetime")))
)

antes_do_corte = col("pickup_date") < lit('2018-01-01')

df_filtrado = (
    df_enriquecido
    .filter(
        antes_do_corte 
        |  ( col("dropOff_datetime") > col("pickup_datetime") ) )
    .filter(
        concat_ws("-",
            year(col("pickup_datetime")).cast("string"),
            lpad(month(col("pickup_datetime")).cast("string"), 2, "0")
        ) == col("_source_file_yearmonth")
    )
    .filter(col("pickup_date") <= '2019-01-31' )
    .filter( 
        antes_do_corte | 
        (
            ( col("DOlocationID") >= 1 ) & ( col("DOlocationID") < 265 ) 
            & (col("PUlocationID") >= 1 ) & ( col("PUlocationID") < 265 )
        )
    )
    .filter( 
        antes_do_corte |
        (
            ( col("DOlocationID") != 264 ) & ( col("PUlocationID") != 264 ) 
            | ( col("DOlocationID") != 265 ) & ( col("PUlocationID") != 265 ) 
        )
    )
    .filter(
        antes_do_corte |
        (
            col("DOlocationID").isNotNull() &
            col("PUlocationID").isNotNull()
        )
    )
)

df_final = (
    df_filtrado.select("pickup_date", "dropoff_date", "pickup_hour", "dropoff_hour",
                       "PUlocationID", "DOlocationID", "dispatching_base_num")
)

df_final.write.mode("overwrite").saveAsTable("mvp.silver.forhirevehicles")

### silver.forhirevehicleshighvolume (High Volume For-Hire Services, HVFHS)

Leitura da tabela `mvp.bronze.forhirevehicleshighvolume`, aplicação dos filtros de qualidade de dados e das transformações de data e hora, e gravação do resultado na tabela Delta `mvp.silver.forhirevehicleshighvolume`. Esta tabela concentra as viagens de aplicativos de alto volume a partir de fev/2019 e é a fonte dos indicadores de tarifa base e remuneração do motorista (perguntas 7 e 8). As justificativas de cada filtro estão detalhadas na seção de Qualidade de Dados.

**Seleção de colunas**: da tabela bronze são mantidas apenas as colunas necessárias às perguntas de negócio: `hvfhs_license_num` (licença da operadora), `pickup_datetime`, `dropoff_datetime`, `PULocationID`, `DOLocationID`, `driver_pay`, `base_passenger_fare`, `tips`, `trip_miles` (usada apenas nos filtros, não persistida na silver) e `_source_file_yearmonth` (não persistida). As demais colunas (taxas, flags de compartilhamento e acessibilidade, bases de despacho, entre outras) são descartadas nesta etapa.

**Transformações aplicadas:**

1. **Separação de data e hora**: cada timestamp é decomposto em uma coluna de data (`pickup_date`, `dropoff_date`) e uma coluna com a hora inteira (`pickup_hour`, `dropoff_hour`), preparando os dados para as agregações por dia e hora da camada gold.

**Filtros de qualidade aplicados** (registros que não atendem a todos são descartados):

1. **Consistência temporal**: o horário de desembarque deve ser posterior ao de embarque.
2. **Valores monetários e distância**: `base_passenger_fare` e `driver_pay` não podem ser negativos, e `trip_miles` deve ser maior que zero, o que exclui as viagens com distância zero (conforme análise descrita na seção de Qualidade de Dados).
3. **Localização desconhecida ou externa**: exclusão de registros com embarque ou desembarque nos códigos 264 ("Desconhecido") e 265 ("Fora de Nova Iorque").
4. **Razão tarifa/distância**: a razão entre a tarifa base e a distância (`base_passenger_fare / trip_miles`) não pode exceder 13,58 dólares por milha.

**Comando final de escrita:**

```python
df_final.write.mode("overwrite").saveAsTable("mvp.silver.forhirevehicleshighvolume")
```

In [0]:
df_fhvhv = spark.table("mvp.bronze.forhirevehicleshighvolume")

df_fhvhv = df_fhvhv.select("hvfhs_license_num", "pickup_datetime", "dropoff_datetime", 
                             "DOLocationID", "PULocationID", "driver_pay", "trip_miles", "base_passenger_fare",
                             "tips", "_source_file_yearmonth")

df_enriquecido = (
    df_fhvhv
    .withColumn("pickup_date", to_date(col("pickup_datetime")))
    .withColumn("dropoff_date", to_date(col("dropoff_datetime")))
    .withColumn("pickup_hour", hour(col("pickup_datetime")))
    .withColumn("dropoff_hour", hour(col("dropOff_datetime")))
)

df_filtrado = (
    df_enriquecido
    .filter( col("dropoff_datetime") > col("pickup_datetime"))
    .filter( (col("base_passenger_fare") >= 0) & ( col("driver_pay") >= 0) & (col("trip_miles") > 0) )
    .filter( (col("DOLocationID") != 264) & (col("PULocationID") != 264) | (col("DOLocationID") != 265) & (col("PULocationID") != 265))
    .filter( try_divide( col("base_passenger_fare"), col("trip_miles")) <= 13.58)
)

df_final = (
    df_filtrado.select("pickup_date", "dropoff_date", "pickup_hour", "dropoff_hour", "hvfhs_license_num",
                       "PULocationID", "DOLocationID", "driver_pay", "base_passenger_fare",
                       "tips")
)

df_final.write.mode("overwrite").saveAsTable("mvp.silver.forhirevehicleshighvolume")

Replica a tabela bronze na camada `silver`.

In [0]:
df_taxiZone = spark.table("mvp.bronze.taxi_zone_lookup")

df_taxiZone.write.mode("overwrite").saveAsTable("mvp.silver.taxi_zone_lookup")

Replica a tabela bronze na camada `silver`.

In [0]:
df_fhv_lookup = spark.table("mvp.bronze.fhv_base_lookup")

df_fhv_lookup.write.mode("overwrite").saveAsTable("mvp.silver.fhv_base_lookup")

### silver.feriados (Feriados Observados do Estado de Nova Iorque)

Leitura da tabela `mvp.bronze.feriados_us_ny`, tratamento dos feriados que possuem data oficial e data de observância distintas, e gravação do resultado na tabela Delta `mvp.silver.feriados`, que contém apenas as datas em que os feriados foram de fato observados. Esta tabela alimenta a flag de feriado e de dia útil/não útil da dimensão `dim_calendario`, conforme descrito na seção de Modelagem.

**Contexto do problema**: a biblioteca `holidays` retorna, para alguns feriados, dois registros: um com a data oficial e outro com a data de observância, identificado pelo sufixo "(Observed)" no nome (ex: quando um feriado cai em um fim de semana, o dia útil observado é a sexta anterior ou a segunda seguinte). Manter os dois registros marcaria como feriado uma data em que não houve, de fato, a folga.

**Transformações e regras aplicadas:**

1. **Identificação de datas observadas**: a coluna `is_observed` sinaliza os registros cujo nome contém "(Observed)" (busca sem distinção entre maiúsculas e minúsculas).
2. **Normalização do nome**: `holiday_name_clean` remove o sufixo "(Observed)" do nome do feriado, permitindo agrupar a data oficial e a data observada do mesmo feriado.
3. **Agrupamento por feriado e ano**: os registros são agrupados por ano (`holiday_year`) e nome normalizado do feriado.
4. **Seleção da data efetiva**: para cada grupo, se existir um registro "observed", apenas ele é mantido (a data oficial é descartada); se não existir, o registro original é mantido normalmente.
5. **Seleção de coluna**: a tabela final mantém apenas a coluna `date`.

**Comando final de escrita:**

```python
df_final.write.mode("overwrite").saveAsTable("mvp.silver.feriados")
```

In [0]:
df_feriado = spark.table("mvp.bronze.feriados_us_ny")

df_flag = (
    df_feriado
    .withColumn("is_observed", col("holiday_name").rlike("(?i)\\(observed\\)"))
    .withColumn(
        "holiday_name_clean",
        trim(regexp_replace(col("holiday_name"), "(?i)\\s*\\(observed\\)", ""))
    )
    .withColumn("holiday_year", year("date"))
)

w = Window.partitionBy("holiday_year", "holiday_name_clean")

df_final = (
    df_flag
    .withColumn("has_observed_in_group", max(col("is_observed").cast("int")).over(w))
    .filter(
        (col("has_observed_in_group") == 0) |
        (col("is_observed") == True)
    )
    .select("date")
)

df_final.write.mode("overwrite").saveAsTable("mvp.silver.feriados")

### silver.inflacao_cpi_u e silver.inflacao_cpi_t (Índices de Inflação)

Leitura das tabelas `mvp.bronze.inflacao_cpi_u` e `mvp.bronze.inflacao_cpi_t`, imputação dos valores mensais ausentes e gravação do resultado nas tabelas Delta `mvp.silver.inflacao_cpi_u` e `mvp.silver.inflacao_cpi_t`. As duas tabelas recebem exatamente o mesmo tratamento, diferindo apenas na coluna do índice: `CPIAUCSL` (inflação geral) e `CPITRNSL` (inflação de transportes). Elas sustentam as perguntas 7 e 8.

**Contexto do problema**: as séries originais possuem meses sem valor de índice (`NULL`). Como os índices serão usados para calcular variações ao longo do tempo e comparar com a evolução de tarifas e da remuneração do motorista, meses ausentes quebrariam o cálculo de variação acumulada. A causa provável é a ausência de coleta em outubro de 2025 (paralisação do governo americano), embora o tratamento abaixo seja genérico e se aplique a qualquer mês ausente.

**Transformações e regras aplicadas:**

1. **Recuperação dos valores vizinhos**: com uma janela ordenada por `observation_date`, são obtidos, para cada mês, o valor do mês anterior (`lag`) e o do mês seguinte (`lead`).
2. **Sinalização da imputação**: a coluna `fl_imputado` marca com `true` os meses cujo valor original era nulo, preservando a rastreabilidade de quais valores foram estimados e quais são dados originais.
3. **Imputação por interpolação linear**: para os meses nulos, o valor do índice é substituído pela média entre o mês anterior e o mês seguinte. Meses com valor original são mantidos sem alteração.
4. **Seleção de colunas**: cada tabela final mantém apenas `observation_date`, o índice (`CPIAUCSL` ou `CPITRNSL`) e `fl_imputado`.

**Comandos finais de escrita:**

```python
df_final.write.mode("overwrite").saveAsTable("mvp.silver.inflacao_cpi_u")
df_final.write.mode("overwrite").saveAsTable("mvp.silver.inflacao_cpi_t")
```

In [0]:
df_cpiu = spark.table("mvp.bronze.inflacao_cpi_u")

w = Window.orderBy("observation_date")

df_final = (
    df_cpiu
    .withColumn("val_anterior", lag("CPIAUCSL", 1).over(w))
    .withColumn("prox_val", lead("CPIAUCSL", 1).over(w))
    .withColumn("fl_imputado", col("CPIAUCSL").isNull())
    .withColumn(
        "CPIAUCSL",
        when(
            col("CPIAUCSL").isNull(),
            (col("val_anterior") + col("prox_val")) / 2
        ).otherwise(col("CPIAUCSL"))
    )
    .select("observation_date", "CPIAUCSL", "fl_imputado")
)

df_final.write.mode("overwrite").saveAsTable("mvp.silver.inflacao_cpi_u")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
df_cpiu = spark.table("mvp.bronze.inflacao_cpi_t")

w = Window.orderBy("observation_date")

df_final = (
    df_cpiu
    .withColumn("val_anterior", lag("CPITRNSL", 1).over(w))
    .withColumn("prox_val", lead("CPITRNSL", 1).over(w))
    .withColumn("fl_imputado", col("CPITRNSL").isNull())
    .withColumn(
        "CPITRNSL",
        when(
            col("CPITRNSL").isNull(),
            (col("val_anterior") + col("prox_val")) / 2
        ).otherwise(col("CPITRNSL"))
    )
    .select("observation_date", "CPITRNSL", "fl_imputado")
)

df_final.write.mode("overwrite").saveAsTable("mvp.silver.inflacao_cpi_t")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
